# Dynamic flux balance analysis (dFBA)

This tutorial builds and simulates a dFBA model of a batch culture of _Escherichia coli_ (genome-scale model iJR904) growing on a mixture of glucose and xylose. Oxygen is consumed during the run, so the culture moves from aerobic to anaerobic growth.

The notebook re-implements [example 1 of the dfba documentation](https://dynamic-fba.readthedocs.io/en/latest/example1.html) with plain cobrapy and `scipy.integrate.solve_ivp`. The `dfba` package itself only has builds for Python 3.8 and older and cannot be installed on current Python versions (including Google Colab).

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you should be able to:

1. Write down a dFBA model as a system of ODEs coupled to an FBA problem (static optimization approach).
2. Express substrate uptake limits with Michaelis-Menten kinetics, including inhibition terms.
3. Simulate a batch culture and interpret diauxic growth, oxygen depletion and ethanol overflow in the results.
4. Modify the model (death rate, initial conditions, substrates, feeding) and predict the effect before running it.

**Estimated time.** 30 min for the walk-through, 15 min for the exercises.

**Prerequisites.** Notebook 01 (FBA with cobrapy) and notebook 05 (changing bounds and the medium). Basic idea of an ordinary differential equation.

### Background

**Why dynamic?** FBA assumes a metabolic steady state and returns *rates* (fluxes, growth rate) for fixed exchange bounds. In a batch culture the extracellular concentrations change over time, and so do the uptake rates the cell can achieve. dFBA couples the two time scales: intracellular metabolism is assumed to be at steady state at every instant (it is much faster), while the extracellular environment is described by ODEs ([Mahadevan et al., 2002](https://doi.org/10.1016/s0006-3495%2802%2973903-9)).

**Static optimization approach (SOA).** At every time point $t$:

1. compute the maximal uptake rates from the current concentrations (the kinetic part),
2. set them as lower bounds of the exchange reactions and solve an FBA problem (maximize growth),
3. use the resulting growth rate $\mu$ and exchange fluxes $v_j$ as the right-hand side of the ODEs

$$
\frac{dX}{dt} = \mu X, \qquad \frac{dC_j}{dt} = v_j X
$$

where $X$ is the biomass concentration, $C_j$ the concentration of extracellular metabolite $j$, and $v_j$ its exchange flux in mmol gDW$^{-1}$ h$^{-1}$ (negative for uptake). The ODE solver (`solve_ivp` with LSODA here) calls this function many times, so a dFBA run is simply hundreds of FBA problems. Because FBA solutions are not always unique, exchange fluxes can jump between alternative optima; the `dfba` package avoids this with lexicographic optimization, which we skip for simplicity.

**Uptake kinetics.** Transport is not described by the genome-scale model, so uptake is limited with Michaelis-Menten kinetics

$$
v_j^{\max}(C_j) = V_{\max,j}\,\frac{C_j}{K_{m,j} + C_j}
$$

multiplied by inhibition terms of the form $1/(1 + C_I/K_I)$. Here ethanol inhibits both sugar uptakes, and glucose inhibits xylose uptake. The last term mimics carbon catabolite repression and produces **diauxic growth**: glucose is used first, xylose only once glucose is (almost) gone. The kinetic parameters come from the dfba example, whose setup resembles the glucose/xylose dFBA model of [Hanly & Henson, 2011](https://doi.org/10.1002/bit.22954) [VERIFY: the dfba documentation cites no source for these values, and they were not checked against that paper].

**Oxygen.** There is no oxygen transfer term, so dissolved oxygen is simply consumed. When it runs out the model switches to anaerobic metabolism and secretes ethanol.

**Units.** Biomass is in gDW L$^{-1}$ and fluxes in mmol gDW$^{-1}$ h$^{-1}$; the factor $1/1000$ in the substrate equations converts mmol to mol. [VERIFY: the dfba documentation does not state the concentration units explicitly; read the numbers qualitatively. A final biomass of about 100 gDW L$^{-1}$ is far above real batch cultures.]

In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

from cobra.io import read_sbml_model

## 1. The genome-scale model

The ODE system we are going to build is

\begin{align}
    \frac{dX}{dt} &= v_{BM}X \\
    \frac{dC_j}{dt} &= v_jX
\end{align}

where $X$ is the biomass concentration (gDW L$^{-1}$), $v_j$ is the flux of exchange reaction $j$ (mmol gDW$^{-1}$ h$^{-1}$) and $v_{BM}$ the flux of the biomass reaction, which equals the growth rate (h$^{-1}$).

First, load the genome-scale metabolic model as a [cobra.Model](https://cobrapy.readthedocs.io/en/latest/building_model.html) object and select GLPK as LP solver.

In [ ]:
fba_model = read_sbml_model("data/iJR904.xml.gz")
fba_model.solver = "glpk"

## 2. Kinetic variables

The state of the ODE system consists of the biomass and the extracellular metabolites we track: **glucose, xylose, oxygen and ethanol**. Glucose, xylose and oxygen get kinetic uptake rules below; ethanol is only secreted, but we track it because it inhibits sugar uptake.

In [ ]:
kinetic_variables = ["Biomass", "Glucose", "Xylose", "Oxygen", "Ethanol"]

## 3. Exchange fluxes

Map each kinetic variable to the reaction in the cobrapy model whose flux changes it. For biomass this is the biomass reaction, for the metabolites the exchange reactions (BiGG identifiers, see [bigg.ucsd.edu](https://bigg.ucsd.edu/models/iJR904)).

In [ ]:
exchange_fluxes = {
    "Biomass": "BIOMASS_Ecoli",
    "Glucose": "EX_glc__D_e",
    "Xylose": "EX_xyl__D_e",
    "Oxygen": "EX_o2_e",
    "Ethanol": "EX_etoh_e",
}

## 4. Right-hand side expressions

Provide a function `(concentrations, fluxes) -> derivative` for each kinetic variable. Compare each line with the ODE system:

\begin{align}
    \frac{dX}{dt} &= v_{BM}X \\
    \frac{dC_j}{dt} &= v_jX
\end{align}

Exchange fluxes are negative for uptake, so substrate concentrations decrease, while secreted products (positive flux) accumulate.

In [ ]:
rhs_expressions = {
    "Biomass": lambda c, v: v["Biomass"] * c["Biomass"],
    "Glucose": lambda c, v: v["Glucose"] * c["Biomass"] / 1000.0,
    "Xylose": lambda c, v: v["Xylose"] * c["Biomass"] / 1000.0,
    "Oxygen": lambda c, v: v["Oxygen"] * c["Biomass"] / 1000.0,
    "Ethanol": lambda c, v: v["Ethanol"] * c["Biomass"] / 1000.0,
}

## 5. Uptake rate expressions

Add functions `concentrations -> maximal uptake rate` for the substrates. The uptake rate is written as a positive number; cobrapy uses negative lower bounds for uptake, so the sign is flipped when the bound is applied (in `fba` below).

Oxygen uptake follows plain Michaelis-Menten kinetics:

$$
v_{O_2}^{\max} = V_{\max}\frac{C_{O_2}}{K_m + C_{O_2}}, \qquad V_{\max} = 15, \; K_m = 0.024
$$

In [ ]:
uptake_rates = {}
uptake_rates["EX_o2_e"] = lambda c: 15.0 * (c["Oxygen"] / (0.024 + c["Oxygen"]))

We are not limited to this formula. Glucose uptake is additionally inhibited by ethanol ($K_I = 20$):

$$
v_{glc}^{\max} = 10.5\,\frac{C_{glc}}{0.0027 + C_{glc}}\cdot\frac{1}{1 + C_{EtOH}/20}
$$

In [ ]:
uptake_rates["EX_glc__D_e"] = lambda c: 10.5 * (c["Glucose"] / (0.0027 + c["Glucose"])) * (1 / (1 + c["Ethanol"] / 20.0))

Xylose uptake is inhibited by both ethanol and glucose. The glucose term has a very small $K_I$ (0.005), so xylose is hardly taken up while glucose is present: this produces the diauxic shift.

In [ ]:
uptake_rates["EX_xyl__D_e"] = lambda c: (
    6.0
    * (c["Xylose"] / (0.0165 + c["Xylose"]))
    * (1 / (1 + c["Ethanol"] / 20.0))
    * (1 / (1 + c["Glucose"] / 0.005))
)

## 6. Simulate

Three small functions do the work:

* `fba(concentrations)` applies the uptake bounds, maximizes growth and returns the exchange fluxes (all zero if the model cannot grow, e.g. when every substrate is used up);
* `rhs(t, y)` turns the state vector `y` into a dictionary, calls `fba` and evaluates the right-hand side expressions;
* `simulate(...)` integrates the ODEs with `scipy.integrate.solve_ivp` and collects the results in two [pandas.DataFrame](https://pandas.pydata.org/)s: concentrations over time, and the fluxes of the exchange reactions recomputed at each stored time point.

The simulation covers $[0, 16]$ hours with results stored every $0.1$ hours. It takes a few seconds.

In [ ]:
def fba(concentrations):
    """Fluxes of `exchange_fluxes` for the given concentrations (all zero if the model can't grow)."""
    concentrations = {k: max(v, 0.0) for k, v in concentrations.items()}  # the integrator may overshoot slightly below 0
    with fba_model:
        for rxn_id, rate in uptake_rates.items():
            fba_model.reactions.get_by_id(rxn_id).lower_bound = -rate(concentrations)
        growth = fba_model.slim_optimize(error_value=0.0)
        return {var: fba_model.reactions.get_by_id(rxn_id).flux if growth > 0 else 0.0
                for var, rxn_id in exchange_fluxes.items()}


def rhs(t, y):
    c = dict(zip(kinetic_variables, np.maximum(y, 0)))
    v = fba(c)
    return [rhs_expressions[var](c, v) for var in kinetic_variables]


def simulate(initial_conditions, t_end, t_step):
    times = np.arange(0.0, t_end + t_step / 2, t_step)
    sol = solve_ivp(rhs, (0.0, t_end), [initial_conditions[var] for var in kinetic_variables],
                    t_eval=times, method="LSODA")
    concentrations = pd.DataFrame(sol.y.T, index=pd.Index(sol.t, name="time"), columns=kinetic_variables)
    trajectories = pd.DataFrame([fba(dict(row)) for _, row in concentrations.iterrows()], index=concentrations.index)
    trajectories = trajectories.rename(columns=exchange_fluxes)
    return concentrations, trajectories


initial_conditions = {
    "Biomass": 0.03,
    "Glucose": 4.0,
    "Xylose": 5.0,
    "Oxygen": 0.5,
    "Ethanol": 0.0,
}
concentrations, trajectories = simulate(initial_conditions, 16.0, 0.1)
trajectories = trajectories[["EX_glc__D_e", "EX_xyl__D_e", "EX_etoh_e"]]

## 7. Plotting the results

The results are plain DataFrames, so any plotting library works; here we use [plotly](https://plotly.com/python/) (`concentrations.plot()` gives the matplotlib equivalent).

In [ ]:
import plotly.express as px

In [ ]:
fig = px.line(concentrations, labels={"value": "concentration", "variable": ""})
fig.show()

Things to look for in the concentration plot:

* **Exponential phase on glucose** while oxygen is still available.
* **Oxygen depletion** (around 9 h): growth slows down and ethanol starts to accumulate, because without oxygen the cell regenerates NAD⁺ by fermentation.
* **Diauxic shift**: xylose is consumed only after glucose is gone (around 15 h).

In [ ]:
fig = px.line(trajectories, labels={"value": "flux [mmol gDW^-1 h^-1]", "variable": ""})
fig.show()

The flux plot shows the same story from the cell's perspective: the glucose uptake rate stays near its maximum until glucose runs out, ethanol secretion switches on when oxygen is depleted, and xylose uptake takes over at the end. Small jumps in the fluxes are alternative optimal FBA solutions (see Background).

## Exercises (15 min)

1. Add a death rate $\delta$ to the model and compare the biomass curves.
    $$
    \frac{dX}{dt} = v_{BM}X - \delta X
    $$
2. Run the simulation under anaerobic conditions from the start. What changes in growth and ethanol production?
3. Replace xylose with another carbon source. Can you add a constant feed (refreshment) $\epsilon$ of this carbon source?
    $$
    \frac{dC_j}{dt} = v_{j}X + \epsilon
    $$

Tip: `rhs`, `fba` and `simulate` look up `kinetic_variables`, `exchange_fluxes`, `rhs_expressions` and `uptake_rates` when they run, so you can change these objects and call `simulate` again.

In [ ]:
# Your code here


## References

- Hanly, Henson (2011). Dynamic flux balance modeling of microbial co‐cultures for efficient batch fermentation of glucose and xylose mixtures. *Biotechnology and Bioengineering* 108:376–385. [doi.org/10.1002/bit.22954](https://doi.org/10.1002/bit.22954)
- Mahadevan, Edwards, Doyle (2002). Dynamic Flux Balance Analysis of Diauxic Growth in Escherichia coli. *Biophysical Journal* 83:1331–1340. [doi.org/10.1016/s0006-3495%2802%2973903-9](https://doi.org/10.1016/s0006-3495%2802%2973903-9)